# Extract Patron Data

Extracts Patron assets and their milestones, effects buffs and targets from Anno 117
into CSV and JSON files. Output goes to `results/tables/`.

Run all cells from the project root.

In [1]:
from pathlib import Path
import json
import re

import pandas as pd

from assetextractor.extraction.utils import Config
from assetextractor.parsing.core.assets import AssetCache

## Load assets

Setup game data for processing.

In [2]:
config = Config.from_json("config.json")
assets = AssetCache.load(config)
templates = assets.templates

print(f"Total assets: {len(assets.elements)}")
print(f"Total texts: {len(assets.texts.elements)}")

Total assets: 30713
Total texts: 33147


In [3]:
asset = assets[37553] # Troop Roman Celtic Auxilia
raw_list = asset.find_value("Maintenance.Maintenances")
print("Asset 37553")
for item in raw_list:
    product = item.find_value("Product")
    print(product, product.__class__.__name__)

asset = assets[43664] # Troop Roman Murmillo Gladiator
print("Asset 43664", asset.find_value("Maintenance.Maintenances"))

Asset 37553
Denarii Asset
Wader Workforce Asset
Plebeian Workforce Asset
Equites Workforce Asset
Patrician Workforce Asset
Asset 43664 [Item (0), Item (1), Item (2), Item (3), Item (4)]


## Extract the Patron assets

Extract the patrons from `Patron` (8 assets).

In [4]:
# Misc: This allow reloading .py modules into jupyter.
%load_ext autoreload
%autoreload 2

In [35]:
from assetextractor.conversion.statistics.patron_extractor import PatronExtractor

extractor = PatronExtractor(assets)
# extractor_de = PatronExtractor(assets, language="german")
patrons_data = extractor.extract_all()

## Print Patrons and their details

Use `extractor.print_patrons()` to see everything, or `extractor.print_patrons(guid=80562)` (Mars as example) to inspect a specific one.

In [32]:
mars_guid = 80562
mars_patron = patrons_data[mars_guid]

print(mars_patron.canonical_name)
print(mars_patron.icon.canonical_name)

patron_mars
icon_patron_mars


In [33]:
extractor.print_patrons(80562)


PATRON: PatronMars (GUID: 80562)
Title:       Armamentum
Description: Increased production of
Asset GUID:  80565
--------------------------------------------------
Buffs: 1
  |- 1 Buff - Buff Mars Production (GUID: 80566)
--------------------------------------------------
Targets: 1
  |- 1 Target: Mars Goods (GUID: 80564)
    |- 1 Target: Production Pasture Roman Celtic Pigs (GUID: 5974)
       [Costs]: 0 Denarii, 1 Timber, 2 Tiles, 0 Concrete
    |- 2 Target: Production Pasture Roman Pigs (GUID: 2793)
       [Costs]: 0 Denarii, 1 Timber, 2 Tiles, 0 Concrete, 0 Marble, 0 Mosaics
    |- 3 Target: Production Mountain Roman Celtic Iron Ore (GUID: 5982)
       [Costs]: 0 Denarii, 2 Timber, 2 Tiles, 0 Concrete
    |- 4 Target: Production Mountain Roman Iron Ore (GUID: 2918)
       [Costs]: 0 Denarii, 2 Timber, 2 Tiles, 0 Concrete, 0 Marble, 0 Mosaics
    |- 5 Target: Production Coast Roman Salt (GUID: 2957)
       [Costs]: 0 Denarii, 2 Timber, 2 Tiles, 0 Concrete, 0 Marble, 0 Mosaics
    |

## Export to JSON

Using the `save_to_json` method from the extractor. You can pass a custom `web_base_path` to fit your webapp asset structure. If left empty, it will provide the original image game path within the asset extractor.

In [38]:
# Setup output directory
output_dir = Path("results/tables")
output_dir.mkdir(parents=True, exist_ok=True)

# --- Scenario 1: Default (Original Game Paths) ---
# This will result in "image_url": "data/ui/4k/base/..." 
target_json_default = output_dir / "patrons_en_original.json"
extractor.save_to_json(
    file_path=target_json_default, 
    web_base_path=None  # Explicitly None to use original paths
)

# --- Scenario 2: Web-Ready (Flattened Paths) ---
# This will result in "image_url": "icons/patrons/icon_patron_mars.webp"
# This matches the output structure of IconProcessor.export_icons
target_json_web = output_dir / "patrons_en_web.json"
extractor.save_to_json(
    file_path=target_json_web, 
    web_base_path="assets/icons/patrons" 
)

print(f"Scenario 1 saved to: {target_json_default}")
print(f"Scenario 2 saved to: {target_json_web}")

Successfully exported 8 ornaments to results\tables\patrons_en_original.json
Successfully exported 8 ornaments to results\tables\patrons_en_web.json
Scenario 1 saved to: results\tables\patrons_en_original.json
Scenario 2 saved to: results\tables\patrons_en_web.json


## Image Export

Convert every icon from .DDS to .webp using wand + magick

In [9]:
print(extractor.patrons.items())

dict_items([(80562, Mars), (43594, Ceres), (27899, Neptune), (50311, Mercury-Lugus), (80861, Epona), (50242, Cernunnos), (27900, Minerva), (144800, Vulcan)])


In [39]:
from assetextractor.conversion.statistics.icon_processor import IconProcessor

patron_list = list(patrons_data.values())

# 2. Run the batch export with compression
IconProcessor.export_icons(
    assets=patron_list,
    output_base="results/icons/patrons", # Custom output folder
    quality=75,                          # Lower quality for better compression
    resize=(128, 128)                    # Resize to small icons
)

  [OK] PatronMars -> results\icons\patrons\icon_patron_mars.webp
  [OK] PatronCeres -> results\icons\patrons\icon_patron_ceres.webp
  [OK] PatronNeptun -> results\icons\patrons\icon_patron_neptune.webp
  [OK] PatronMercury -> results\icons\patrons\icon_patron_mercury_lugus.webp
  [OK] PatronEpona -> results\icons\patrons\icon_patron_epona.webp
  [OK] PatronCernunnos -> results\icons\patrons\icon_patron_cernunnos.webp
  [OK] PatronMinerva -> results\icons\patrons\icon_patron_minerva.webp
  [OK] PatronVulcanus -> results\icons\patrons\icon_patron_vulcan.webp
---
Finished! Exported: 8 | Skipped: 0


{'exported': 8, 'skipped': 0, 'errors': 0}